# The Intender Gap — Data Preprocessing
## `02_preprocessing.ipynb`

---

### Business Question

**How do we prepare the raw survey data so the model can learn from it — and what choices do we make along the way?**

The Stack Overflow 2025 survey gives us 49,196 raw rows and dozens of columns. Before any model can run, three things have to happen:

1. **Filter** — keep only the rows and columns that are relevant to the intender-vs-rejector question
2. **Clean** — handle multi-select columns, ordinal categories, missing values, and class imbalance in a principled way
3. **Split** — divide into training and test sets so model evaluation is honest

Every decision in this notebook has a documented reason. The goal is not just clean data — it is a feature matrix that a PM can look at and say: *"I understand why each of these variables is here."*

**Key decisions from EDA already locked in (see DECISIONS.md D-13):**
- `WorkExp` dropped — r = 0.844 with `YearsCode`, multicollinearity risk too high for Logistic Regression
- `AIComplex` included — missingness check confirmed above 60% non-null threshold for both intenders and rejectors
- `AIFrustration` included — same result; non-users have formed opinions through secondhand experience
- `ICorPM` and `PurchaseInfluence` included but flagged as likely low importance — Random Forest will confirm

---
## Section 1 — Library Imports and Path Setup

Standard libraries only — no new packages beyond what is listed in CLAUDE.md.
`pathlib.Path` is used throughout so paths work correctly regardless of
whether the notebook is run from the `notebooks/` directory or the repo root.

In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Consistent group color scheme (matches 01_eda.ipynb) ──────────────────────
GROUP_COLORS = {
    'active':   '#1D9E75',   # teal
    'casual':   '#BA7517',   # amber
    'intender': '#534AB7',   # purple
    'rejector': '#993C1D',   # coral
}
GROUP_ORDER  = ['active', 'casual', 'intender', 'rejector']
GROUP_LABELS = {
    'active':   'Active Users',
    'casual':   'Casual Users',
    'intender': 'Intenders',
    'rejector': 'Rejectors',
}

# ── File paths ─────────────────────────────────────────────────────────────────
# Notebook is run from the notebooks/ directory, so data is one level up
RAW_DATA_PATH      = Path('../data/raw/survey_results_public.csv')
PROCESSED_DIR      = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DF_VALID_PATH      = PROCESSED_DIR / 'df_valid.csv'
DF_MODEL_PATH      = PROCESSED_DIR / 'df_intenders_rejectors.csv'
FEATURE_NAMES_PATH = PROCESSED_DIR / 'feature_names.txt'

print('Paths confirmed:')
print(f'  Raw data:  {RAW_DATA_PATH.resolve()}')
print(f'  Processed: {PROCESSED_DIR.resolve()}')

Paths confirmed:
  Raw data:  C:\Debjyoti\01 Personal\06 Claude\Data Science Projects\developer-ai-adoption-gap\data\raw\survey_results_public.csv
  Processed: C:\Debjyoti\01 Personal\06 Claude\Data Science Projects\developer-ai-adoption-gap\data\processed


---
## Section 2 — Load Raw Data, Filter to Valid Responses, Assign Group Labels

**Why we filter first:** 15,471 rows have a null `AISelect` value. These are
survey dropouts who never reached the AI section — confirmed as Missing Completely
at Random (MCAR) via dropout curve analysis in EDA (see DECISIONS.md D-07).
We drop them here and never touch them again.

**Why we create group labels:** The raw `AISelect` column stores the full survey
response text. A shorter group label (`active`, `casual`, `intender`, `rejector`)
is easier to work with in code and consistent with `01_eda.ipynb`.

**Why we filter further to intenders and rejectors:** The primary model only needs
these two groups. Both are non-users — holding AI tool access and usage habit
constant, the only meaningful difference between them is intent. That is exactly
what the model is trying to explain (see DECISIONS.md D-08).

In [2]:
# ── Step 1: Load the raw survey file ──────────────────────────────────────────
df_raw = pd.read_csv(RAW_DATA_PATH, low_memory=False)
print(f'Raw dataset loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')

# ── Step 2: Drop rows where AISelect is null (MCAR — see DECISIONS.md D-07) ───
df_valid = df_raw[df_raw['AISelect'].notna()].copy()
rows_dropped = len(df_raw) - len(df_valid)
print(f'\nRows after dropping AISelect nulls: {len(df_valid):,}')
print(f'Rows dropped (MCAR dropouts):       {rows_dropped:,}')

# ── Step 3: Map raw AISelect text to four group labels ────────────────────────
aiselect_to_group = {
    'Yes, I use AI tools daily':                   'active',
    'Yes, I use AI tools weekly':                  'active',
    'Yes, I use AI tools monthly or infrequently': 'casual',
    'No, but I plan to soon':                      'intender',
    "No, and I don't plan to":                     'rejector',
}

df_valid['group'] = df_valid['AISelect'].map(aiselect_to_group)

# Confirm all AISelect values were mapped (no nulls in 'group' after mapping)
unmapped = df_valid['group'].isna().sum()
assert unmapped == 0, f'Unexpected: {unmapped} rows could not be mapped to a group'

print('\nGroup assignment counts:')
group_counts = df_valid['group'].value_counts().reindex(GROUP_ORDER)
group_pct    = (group_counts / len(df_valid) * 100).round(1)
summary = pd.DataFrame({'Count': group_counts, 'Percent (%)': group_pct})
print(summary.to_string())

# ── Step 4: Save df_valid for EDA reference ───────────────────────────────────
df_valid.to_csv(DF_VALID_PATH, index=False)
print(f'\ndf_valid saved to: {DF_VALID_PATH}')

# ── Step 5: Filter to intenders and rejectors only (primary model subset) ─────
df_model = df_valid[df_valid['group'].isin(['intender', 'rejector'])].copy()

print(f'\nModel subset (intenders + rejectors):')
print(f'  Total rows: {len(df_model):,}')
print(f'  Intenders:  {(df_model["group"] == "intender").sum():,}  '
      f'({(df_model["group"] == "intender").mean() * 100:.1f}%)')
print(f'  Rejectors:  {(df_model["group"] == "rejector").sum():,}  '
      f'({(df_model["group"] == "rejector").mean() * 100:.1f}%)')
print(f'\nClass split confirms ~25/75 imbalance — class_weight="balanced" required on all models.')

Raw dataset loaded: 49,191 rows x 172 columns

Rows after dropping AISelect nulls: 33,720
Rows dropped (MCAR dropouts):       15,471

Group assignment counts:
          Count  Percent (%)
group                       
active    21841         64.8
casual     4628         13.7
intender   1797          5.3
rejector   5454         16.2



df_valid saved to: ..\data\processed\df_valid.csv

Model subset (intenders + rejectors):
  Total rows: 7,251
  Intenders:  1,797  (24.8%)
  Rejectors:  5,454  (75.2%)

Class split confirms ~25/75 imbalance — class_weight="balanced" required on all models.


---
## Section 3 — Explode Multi-Select Columns into Binary Indicators

`DevType` and `AIFrustration` store multiple choices as a single semicolon-separated
string per row (e.g. `"Developer, full-stack;Developer, back-end"`). A model cannot
learn from a string — we need one binary column per unique choice.

**Why a 5% frequency threshold?** After explosion, rare categories (e.g. a job title
held by fewer than 1 in 20 respondents) add columns without adding signal. They also
create near-zero-variance features that can destabilise Logistic Regression. Dropping
anything below 5% keeps the feature set manageable without discarding meaningful roles
or frustrations. The threshold is applied to the intender/rejector subset only — these
are the only rows the model will ever see.

**Naming convention:** `DevType_FullStackDev`, `AIFrustration_AlmostRight`, etc.
Raw survey text is shortened to snake-case-style labels so column names are readable
in feature importance charts without truncation.

In [3]:
def explode_multiselect(df, column, prefix, label_map, min_pct=0.05):
    """
    Convert a semicolon-separated multi-select column into binary indicator columns.

    Parameters
    ----------
    df         : DataFrame containing the column
    column     : name of the semicolon-separated column to explode
    prefix     : string prefix for new column names (e.g. 'DevType')
    label_map  : dict mapping raw survey text to short column-safe labels
    min_pct    : minimum fraction of rows that must have a value to keep it (default 5%)

    Returns
    -------
    df         : DataFrame with binary indicator columns added, original column dropped
    kept       : list of (col_name, frequency) tuples that passed the threshold
    dropped    : list of (col_name, frequency) tuples that were below the threshold
    """
    # Split each row's semicolon string into a list; NaN rows become empty lists
    split_series = df[column].fillna('').str.split(';')

    # Get all unique non-empty values across every row
    all_values = sorted({
        val.strip()
        for val_list in split_series
        for val in val_list
        if val.strip() != ''
    })

    kept    = []
    dropped = []

    for raw_value in all_values:
        # Create binary indicator: 1 if this value appears in the row's list
        indicator = split_series.apply(
            lambda val_list: 1 if raw_value in [v.strip() for v in val_list] else 0
        )
        freq = indicator.mean()

        # Map raw text to a short readable label; fall back to raw text if not in map
        short_label = label_map.get(raw_value, raw_value)
        col_name    = f'{prefix}_{short_label}'

        if freq >= min_pct:
            df[col_name] = indicator
            kept.append((col_name, freq))
        else:
            dropped.append((col_name, freq))

    # Drop the original semicolon column — indicators replace it
    df = df.drop(columns=[column])

    return df, kept, dropped


# ── Label maps: raw survey text → short column-safe name ──────────────────────
# Note: AIFrustration values use Unicode curly apostrophes (\u2019) in the raw
# data — keys below match exactly. Verified against raw CSV before coding.

DEVTYPE_LABEL_MAP = {
    'Developer, full-stack':                          'FullStack',
    'Developer, back-end':                            'Backend',
    'Developer, front-end':                           'Frontend',
    'Developer, desktop or enterprise applications':  'DesktopEnterprise',
    'Developer, mobile':                              'Mobile',
    'Developer, embedded applications or devices':    'Embedded',
    'Developer, game or graphics':                    'GameGraphics',
    'DevOps engineer or professional':                'DevOps',
    'Data scientist':                                 'DataScientist',
    'Data engineer':                                  'DataEngineer',
    'Data or business analyst':                       'DataAnalyst',
    'AI/ML engineer':                                 'MLEngineer',
    'Engineering manager':                            'EngineeringManager',
    'Architect, software or solutions':               'Architect',
    'System administrator':                           'SysAdmin',
    'Academic researcher':                            'AcademicResearcher',
    'Student':                                        'Student',
    'Retired':                                        'Retired',
    'Applied scientist':                              'AppliedScientist',
    'Support engineer or analyst':                    'SupportEngineer',
    'Cybersecurity or InfoSec professional':          'SecurityPro',
    'Cloud infrastructure engineer':                  'CloudInfra',
    'Database administrator or engineer':             'DBAEngineer',
    'Developer, QA or test':                          'QATest',
    'Developer, AI apps or physical AI':              'AIDev',
    'Senior executive (C-suite, VP, etc.)':           'CxO',
    'Project manager':                                'ProjectManager',
    'Product manager':                                'ProductManager',
    'Founder, technology or otherwise':               'Founder',
    'Financial analyst or engineer':                  'FinancialAnalyst',
    'UX, Research Ops or UI design professional':     'UXDesigner',
    'Other (please specify):':                        'Other',
}

AIFRUSTRATION_LABEL_MAP = {
    'AI solutions that are almost right, but not quite':         'AlmostRight',
    'Debugging AI-generated code is more time-consuming':        'DebuggingSlower',
    'It\u2019s hard to understand how or why the code works':    'HardToUnderstand',
    'I\u2019ve become less confident in my own problem-solving': 'LessConfident',
    'I don\u2019t use AI tools regularly':                       'DontUseRegularly',
    'I haven\u2019t encountered any problems':                   'NoProblems',
    'Other (write in):':                                         'Other',
}

# ── Print shape before explosion ───────────────────────────────────────────────
shape_before = df_model.shape
print(f'Shape BEFORE explosion: {shape_before[0]:,} rows x {shape_before[1]} columns')
print()

# ── Explode DevType ────────────────────────────────────────────────────────────
df_model, devtype_kept, devtype_dropped = explode_multiselect(
    df_model, 'DevType', 'DevType', DEVTYPE_LABEL_MAP, min_pct=0.05
)

print(f'DevType — {len(devtype_kept)} indicators kept (>= 5% frequency):')
for col, freq in sorted(devtype_kept, key=lambda x: -x[1]):
    print(f'    {col:<40}  {freq * 100:.1f}%')
print(f'\nDevType — {len(devtype_dropped)} indicators dropped (< 5% frequency):')
for col, freq in sorted(devtype_dropped, key=lambda x: -x[1]):
    print(f'    {col:<40}  {freq * 100:.1f}%')
print()

# ── Explode AIFrustration ──────────────────────────────────────────────────────
df_model, frust_kept, frust_dropped = explode_multiselect(
    df_model, 'AIFrustration', 'AIFrustration', AIFRUSTRATION_LABEL_MAP, min_pct=0.05
)

print(f'AIFrustration — {len(frust_kept)} indicators kept (>= 5% frequency):')
for col, freq in sorted(frust_kept, key=lambda x: -x[1]):
    print(f'    {col:<50}  {freq * 100:.1f}%')
print(f'\nAIFrustration — {len(frust_dropped)} indicators dropped (< 5% frequency):')
for col, freq in sorted(frust_dropped, key=lambda x: -x[1]):
    print(f'    {col:<50}  {freq * 100:.1f}%')
print()

# ── Print shape after explosion ────────────────────────────────────────────────
shape_after = df_model.shape
cols_added  = (len(devtype_kept) + len(frust_kept))   # indicator columns created
cols_removed = 2                                        # DevType + AIFrustration originals dropped
print(f'Shape AFTER explosion:  {shape_after[0]:,} rows x {shape_after[1]} columns')
print(f'  Indicator columns added:          {cols_added}')
print(f'  Original columns removed:         {cols_removed}')
print(f'  Net column change:                +{cols_added - cols_removed}')

Shape BEFORE explosion: 7,251 rows x 173 columns



DevType — 6 indicators kept (>= 5% frequency):
    DevType_FullStack                         23.1%
    DevType_Backend                           12.8%
    DevType_Student                           8.5%
    DevType_DesktopEnterprise                 6.8%
    DevType_Other                             6.3%
    DevType_Architect                         5.3%

DevType — 26 indicators dropped (< 5% frequency):
    DevType_Embedded                          4.9%
    DevType_AcademicResearcher                3.7%
    DevType_Retired                           3.0%
    DevType_Frontend                          2.7%
    DevType_DevOps                            2.4%
    DevType_Mobile                            2.1%
    DevType_GameGraphics                      2.0%
    DevType_SysAdmin                          1.8%
    DevType_EngineeringManager                1.8%
    DevType_DataEngineer                      1.4%
    DevType_AppliedScientist                  1.1%
    DevType_DataScientist        

---
## Section 4 — Ordinal Encoding: Age and EdLevel

`Age` and `EdLevel` are stored as category strings but have a meaningful order
— a 35-year-old has more career experience than a 22-year-old, and a Master's
graduate has more formal education than someone with some college. Using integers
that respect this order lets the model learn from the direction of the relationship,
not just the presence of a category.

**Age mapping (per CLAUDE.md):** 18–24 = 1 through 65+ = 6. `'Prefer not to say'`
is not mappable to any ordinal position → left as NaN and imputed with mode later.

**EdLevel mapping:** Primary school = 1 through Professional degree = 7.
`'Other (please specify):'` has no defined position in the education ladder
→ left as NaN and imputed with mode later.

Note: EdLevel strings in the raw CSV use Unicode curly apostrophes (`\u2019`)
in `Bachelor's` and `Master's` — the map keys below match the exact bytes in
the file, verified before writing this cell.

In [4]:
# ── Age ordinal map (CLAUDE.md spec: 18-24=1 through 65+=6) ───────────────────
age_ordinal_map = {
    '18-24 years old':  1,
    '25-34 years old':  2,
    '35-44 years old':  3,
    '45-54 years old':  4,
    '55-64 years old':  5,
    '65 years or older': 6,
    # 'Prefer not to say' → not in map → NaN → imputed with mode in Section 7
}

# ── EdLevel ordinal map (lowest formal education → highest) ───────────────────
# Keys use Unicode curly apostrophes (\u2019) matching the raw CSV exactly
edlevel_ordinal_map = {
    'Primary/elementary school':                                                         1,
    'Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)': 2,
    'Some college/university study without earning a degree':                             3,
    'Associate degree (A.A., A.S., etc.)':                                               4,
    'Bachelor\u2019s degree (B.A., B.S., B.Eng., etc.)':                                5,
    'Master\u2019s degree (M.A., M.S., M.Eng., MBA, etc.)':                             6,
    'Professional degree (JD, MD, Ph.D, Ed.D, etc.)':                                   7,
    # 'Other (please specify):' → not in map → NaN → imputed with mode in Section 7
}

df_model['Age']     = df_model['Age'].map(age_ordinal_map)
df_model['EdLevel'] = df_model['EdLevel'].map(edlevel_ordinal_map)

print('Age encoding:')
print(df_model['Age'].value_counts(dropna=False).sort_index().to_string())
print(f'\nAge NaN count (from "Prefer not to say"): {df_model["Age"].isna().sum()}')

print('\nEdLevel encoding:')
print(df_model['EdLevel'].value_counts(dropna=False).sort_index().to_string())
print(f'\nEdLevel NaN count (from "Other"): {df_model["EdLevel"].isna().sum()}')

print(f'\nShape unchanged (values remapped in-place): {df_model.shape}')

Age encoding:
Age
1.0    1109
2.0    1996
3.0    1881
4.0    1072
5.0     739
6.0     377
NaN      77

Age NaN count (from "Prefer not to say"): 77

EdLevel encoding:
EdLevel
1.0      68
2.0     602
3.0    1008
4.0     225
5.0    2811
6.0    1871
7.0     526
NaN     140

EdLevel NaN count (from "Other"): 140

Shape unchanged (values remapped in-place): (7251, 183)


---
## Section 5 — Employment: Collapse then Encode

The raw `Employment` column has six categories, but many are functionally equivalent
for our purposes. We collapse to three groups before encoding (per CLAUDE.md):

| Raw value | Collapsed category | Encoded |
|---|---|---|
| Employed | employed_full_time | 0 |
| Independent contractor, freelancer, or self-employed | independent_freelance | 1 |
| Student / Not employed / Retired / I prefer not to say | student_other | 2 |

**Why collapse first, encode second?** Encoding six sparse categories would create
four dummy columns with very few rows in some cells, making coefficients unstable.
The three-way collapse reflects meaningful differences in incentive structure:
full-time employees face org-level adoption barriers, freelancers make their own
tool decisions, and the student/other group has no organisational constraint at all.

In [5]:
def collapse_employment(val):
    """Map raw Employment strings to one of three collapsed categories."""
    if pd.isna(val):
        return np.nan
    if val == 'Employed':
        return 'employed_full_time'
    elif val == 'Independent contractor, freelancer, or self-employed':
        return 'independent_freelance'
    else:
        # Student, Not employed, Retired, I prefer not to say
        return 'student_other'

employment_encode_map = {
    'employed_full_time':  0,
    'independent_freelance': 1,
    'student_other':       2,
}

df_model['Employment'] = (
    df_model['Employment']
    .apply(collapse_employment)
    .map(employment_encode_map)
)

print('Employment encoding (all six raw values collapsed to three):')
print(df_model['Employment'].value_counts(dropna=False).sort_index().to_string())
print('\n  0 = Employed full-time')
print('  1 = Independent / Freelance')
print('  2 = Student / Not employed / Retired / Prefer not to say')
print(f'\nEmployment NaN count: {df_model["Employment"].isna().sum()}')
print(f'Shape unchanged (values remapped in-place): {df_model.shape}')

Employment encoding (all six raw values collapsed to three):
Employment
0    4782
1     947
2    1522

  0 = Employed full-time
  1 = Independent / Freelance
  2 = Student / Not employed / Retired / Prefer not to say

Employment NaN count: 0
Shape unchanged (values remapped in-place): (7251, 183)


---
## Section 6 — Nominal Encoding: LearnCodeAI Binary + get_dummies

**LearnCodeAI** has five response options, two of which start with
`"Yes, I learned how to use AI-enabled tools"` (one for job, one for hobbies).
Both signal active AI learning investment — the distinction between personal and
professional motivation is not relevant to our model. We collapse to a single
binary flag: 1 = actively learned AI tooling, 0 = did not.

**get_dummies** is applied to the eight remaining nominal columns. `drop_first=True`
drops one dummy per column to avoid the dummy variable trap — if a respondent is
not `Remote` and not `Hybrid (in-person)` etc., the model can infer `In-person`
without an explicit column, keeping the feature matrix full-rank for Logistic
Regression.

Columns receiving get_dummies treatment:
`Industry`, `OrgSize`, `ICorPM`, `RemoteWork`, `PurchaseInfluence`,
`AISent`, `AIThreat`, `AIComplex`

NaN rows in these columns receive 0 across all their dummy columns —
the model treats them as "no stated category", which is a reasonable
default given the 5–27% missingness rates observed in EDA.

In [6]:
# ── LearnCodeAI → binary flag ──────────────────────────────────────────────────
# Both "Yes" variants start with the same prefix — one binary column captures both
df_model['LearnCodeAI_flag'] = (
    df_model['LearnCodeAI']
    .fillna('')
    .str.startswith('Yes, I learned how to use AI-enabled tools')
    .astype(int)
)
df_model = df_model.drop(columns=['LearnCodeAI'])

print('LearnCodeAI_flag distribution:')
print(df_model['LearnCodeAI_flag'].value_counts().to_string())
print('  1 = Actively learned AI tooling this past year')
print('  0 = Did not (or no response)')
print()

# ── get_dummies for nominal columns ───────────────────────────────────────────
nominal_cols = [
    'Industry', 'OrgSize', 'ICorPM', 'RemoteWork', 'PurchaseInfluence',
    'AISent', 'AIThreat', 'AIComplex',
]

shape_before_dummies = df_model.shape
df_model = pd.get_dummies(df_model, columns=nominal_cols, drop_first=True)
shape_after_dummies = df_model.shape

dummies_added   = shape_after_dummies[1] - shape_before_dummies[1]
cols_removed    = len(nominal_cols)           # original columns dropped by get_dummies
indicators_created = dummies_added + cols_removed

print(f'get_dummies summary:')
print(f'  Original nominal columns removed: {cols_removed}')
print(f'  Dummy columns created:            {indicators_created}')
print(f'  Net column change:                +{dummies_added}')
print(f'\nShape before get_dummies: {shape_before_dummies[0]:,} rows x {shape_before_dummies[1]} columns')
print(f'Shape after  get_dummies: {shape_after_dummies[0]:,} rows x {shape_after_dummies[1]} columns')

# Show the dummy column names created for each original column (for traceability)
print('\nDummy columns created per original column:')
for col in nominal_cols:
    new_cols = [c for c in df_model.columns if c.startswith(col + '_')]
    print(f'  {col} ({len(new_cols)} dummies): {new_cols}')

LearnCodeAI_flag distribution:
LearnCodeAI_flag
0    5803
1    1448
  1 = Actively learned AI tooling this past year
  0 = Did not (or no response)

get_dummies summary:
  Original nominal columns removed: 8
  Dummy columns created:            43
  Net column change:                +35

Shape before get_dummies: 7,251 rows x 183 columns
Shape after  get_dummies: 7,251 rows x 218 columns

Dummy columns created per original column:
  Industry (14 dummies): ['Industry_Computer Systems Design and Services', 'Industry_Energy', 'Industry_Fintech', 'Industry_Government', 'Industry_Healthcare', 'Industry_Higher Education', 'Industry_Insurance', 'Industry_Internet, Telecomm or Information Services', 'Industry_Manufacturing', 'Industry_Media & Advertising Services', 'Industry_Other:', 'Industry_Retail and Consumer Services', 'Industry_Software Development', 'Industry_Transportation, or Supply Chain']
  OrgSize (8 dummies): ['OrgSize_10,000 or more employees', 'OrgSize_100 to 499 employees', 'Org

---
## Section 7 — Missing Value Imputation

After all encoding steps, three types of NAs remain in the feature columns:

| Column | Source of NAs | Strategy |
|---|---|---|
| `YearsCode` | Raw survey — some respondents skipped | **Median** — right-skewed distribution, robust to outliers |
| `Age` | `'Prefer not to say'` responses (unmappable) | **Mode** — ordinal integer, most common age band |
| `EdLevel` | `'Other (please specify):'` responses (unmappable) | **Mode** — ordinal integer, most common education level |

`Employment` requires no imputation — every raw value maps to one of the three
collapsed categories, so there are no NaNs after encoding.

`YearsCode` is stored as a string in the raw CSV (some respondents write
`"More than 50 years"` or `"Less than 1 year"`). We convert to numeric first
with `pd.to_numeric(errors='coerce')`, which turns non-numeric strings into NaN
before we impute the median. This mirrors the approach used in `01_eda.ipynb`.

Dummy columns from Section 6 have no NAs — `pd.get_dummies` assigns 0 to rows
where the original column was NaN, treating them as "no stated category".

In [7]:
# ── YearsCode: convert to numeric then impute with median ─────────────────────
# pd.to_numeric coerces non-numeric strings (e.g. 'More than 50 years') to NaN
df_model['YearsCode'] = pd.to_numeric(df_model['YearsCode'], errors='coerce')

years_code_na_before = df_model['YearsCode'].isna().sum()
years_code_median    = df_model['YearsCode'].median()
df_model['YearsCode'] = df_model['YearsCode'].fillna(years_code_median)

print(f'YearsCode:')
print(f'  NAs before imputation: {years_code_na_before}')
print(f'  Median used:           {years_code_median:.1f} years')
print(f'  NAs after imputation:  {df_model["YearsCode"].isna().sum()}')
print()

# ── Age: impute with mode (most common ordinal level) ─────────────────────────
age_na_before = df_model['Age'].isna().sum()
age_mode      = df_model['Age'].mode()[0]
df_model['Age'] = df_model['Age'].fillna(age_mode)

print(f'Age:')
print(f'  NAs before imputation: {age_na_before}  (from "Prefer not to say")')
print(f'  Mode used:             {int(age_mode)}  (= {[k for k,v in age_ordinal_map.items() if v == age_mode][0]})')
print(f'  NAs after imputation:  {df_model["Age"].isna().sum()}')
print()

# ── EdLevel: impute with mode (most common ordinal level) ─────────────────────
edlevel_na_before = df_model['EdLevel'].isna().sum()
edlevel_mode      = df_model['EdLevel'].mode()[0]
df_model['EdLevel'] = df_model['EdLevel'].fillna(edlevel_mode)

edlevel_reverse_map = {v: k for k, v in edlevel_ordinal_map.items()}
print(f'EdLevel:')
print(f'  NAs before imputation: {edlevel_na_before}  (from "Other (please specify):")')
print(f'  Mode used:             {int(edlevel_mode)}  (= {edlevel_reverse_map[edlevel_mode]})')
print(f'  NAs after imputation:  {df_model["EdLevel"].isna().sum()}')
print()

# ── Confirm no NAs remain in the columns we encoded ───────────────────────────
encoded_cols = ['YearsCode', 'Age', 'EdLevel', 'Employment']
remaining_na = df_model[encoded_cols].isna().sum()
print('NA check on encoded feature columns after imputation:')
print(remaining_na.to_string())

print(f'\nShape unchanged by imputation: {df_model.shape}')

YearsCode:
  NAs before imputation: 115
  Median used:           17.0 years
  NAs after imputation:  0

Age:
  NAs before imputation: 77  (from "Prefer not to say")
  Mode used:             2  (= 25-34 years old)
  NAs after imputation:  0

EdLevel:
  NAs before imputation: 140  (from "Other (please specify):")
  Mode used:             5  (= Bachelor’s degree (B.A., B.S., B.Eng., etc.))
  NAs after imputation:  0

NA check on encoded feature columns after imputation:
YearsCode     0
Age           0
EdLevel       0
Employment    0

Shape unchanged by imputation: (7251, 218)


---
## Section 8 — Create Binary Target Column and Drop WorkExp

**Binary target (`intender_flag`):** The model needs a numeric 0/1 target.
Intenders (the group we want to identify) = 1; Rejectors (the comparison group) = 0.
This is derived from the `group` column which was created in Section 2.

**Drop `WorkExp`:** Confirmed dropped per DECISIONS.md D-13. Pearson r = 0.844
with `YearsCode` — close enough to the 0.85 multicollinearity threshold that
we prefer to remove it before Logistic Regression rather than rely on VIF to
catch it later. `YearsCode` is retained as the more directly relevant measure
of coding-context experience. We check with `if 'WorkExp' in df_model.columns`
before dropping so the cell is safe to re-run.

In [8]:
# ── Create binary target column ────────────────────────────────────────────────
df_model['intender_flag'] = (df_model['group'] == 'intender').astype(int)

print('intender_flag value counts:')
counts = df_model['intender_flag'].value_counts().sort_index()
pcts   = (counts / len(df_model) * 100).round(1)
summary = pd.DataFrame({'Count': counts, 'Percent (%)': pcts})
summary.index = summary.index.map({1: '1 = Intender', 0: '0 = Rejector'})
print(summary.to_string())
print(f'\nTotal rows: {len(df_model):,}')
print(f'Class split confirms ~25/75 imbalance — class_weight="balanced" required on all models.')

# ── Drop WorkExp (see DECISIONS.md D-13) ──────────────────────────────────────
if 'WorkExp' in df_model.columns:
    df_model = df_model.drop(columns=['WorkExp'])
    print(f'\nWorkExp dropped (r=0.844 with YearsCode — multicollinearity risk).')
else:
    print(f'\nWorkExp not found in columns — already removed or never present.')

print(f'\nShape after target creation and WorkExp drop: {df_model.shape}')

intender_flag value counts:


               Count  Percent (%)
intender_flag                    
0 = Rejector    5454         75.2
1 = Intender    1797         24.8

Total rows: 7,251
Class split confirms ~25/75 imbalance — class_weight="balanced" required on all models.

WorkExp dropped (r=0.844 with YearsCode — multicollinearity risk).

Shape after target creation and WorkExp drop: (7251, 218)


---
## Section 9 — Stratified Train / Test Split (80/20)

Before splitting, we select only the feature columns the model will actually use —
everything except identifiers, raw survey text, and the target itself. The `group`
and `AISelect` columns are labels, not features. Any remaining raw string columns
that were not encoded are also excluded here.

**Why stratified?** With a 25/75 class split, a random split could by chance put
fewer intenders in the test set, making evaluation unreliable. `stratify=y` forces
sklearn to preserve the 24.8/75.2 ratio in both train and test — confirmed by the
class distribution print below.

**Why 80/20?** Standard split for a dataset of this size. 80% gives the model
~5,800 rows to learn from; 20% gives ~1,450 rows for honest evaluation.
Even with 20%, the test set contains ~360 intenders — sufficient for a stable
AUC-ROC estimate.

`random_state=42` is set for full reproducibility — anyone re-running this
notebook gets the identical split.

In [9]:
from sklearn.model_selection import train_test_split

# ── Build the feature list explicitly from the CLAUDE.md feature plan ───────────
# We select by prefix (for dummified/exploded columns) and by exact name
# (for ordinal/numeric columns). Every other raw survey column is excluded.
# This prevents target leakage and eliminates nulls from unencoded columns.

intended_prefixes = [
    "DevType_",           # multi-select explosion
    "AIFrustration_",     # multi-select explosion
    "Industry_",          # get_dummies
    "OrgSize_",           # get_dummies
    "ICorPM_",            # get_dummies
    "RemoteWork_",        # get_dummies
    "PurchaseInfluence_", # get_dummies
    "AISent_",            # get_dummies
    "AIThreat_",          # get_dummies
    "AIComplex_",         # get_dummies
]

intended_exact = [
    "Age",               # ordinal encoded 1-6
    "EdLevel",           # ordinal encoded 1-7
    "Employment",        # collapsed + encoded 0/1/2
    "YearsCode",         # numeric (median imputed)
    "LearnCodeAI_flag",  # binary flag
]

# Select prefix-matched columns in the order they appear in df_model
prefix_cols = [
    c for c in df_model.columns
    if any(c.startswith(p) for p in intended_prefixes)
]
# Add exact-name columns (only if present — WorkExp was already dropped)
exact_cols = [c for c in intended_exact if c in df_model.columns]

feature_cols = prefix_cols + exact_cols

X = df_model[feature_cols]
y = df_model["intender_flag"]

print(f"Feature matrix X: {X.shape[0]:,} rows x {X.shape[1]} columns")
print(f"Target vector  y: {y.shape[0]:,} rows")
print()
print("Features by group:")
for p in intended_prefixes:
    cols = [c for c in feature_cols if c.startswith(p)]
    print(f"  {p:<22} {len(cols)} columns")
print(f"  {chr(39)+'Exact (ordinal/numeric)'+chr(39):<22} {len(exact_cols)} columns  {exact_cols}")
print()

# ── Stratified 80/20 split ─────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,          # preserves 25/75 class ratio in both splits
    random_state=42      # reproducibility
)

# ── Print class distribution in each split ────────────────────────────────
def print_split_distribution(y_split, split_name):
    counts = y_split.value_counts().sort_index()
    pcts   = (counts / len(y_split) * 100).round(1)
    print(f"{split_name} ({len(y_split):,} rows):")
    print(f"  Intenders (1): {counts[1]:>5,}   {pcts[1]}%")
    print(f"  Rejectors (0): {counts[0]:>5,}   {pcts[0]}%")

print_split_distribution(y_train, "Train set")
print()
print_split_distribution(y_test,  "Test set")
print()
print("Stratification confirmed — 25/75 ratio preserved in both splits.")

Feature matrix X: 7,251 rows x 60 columns
Target vector  y: 7,251 rows

Features by group:
  DevType_               6 columns
  AIFrustration_         6 columns
  Industry_              14 columns
  OrgSize_               8 columns
  ICorPM_                1 columns
  RemoteWork_            4 columns
  PurchaseInfluence_     4 columns
  AISent_                5 columns
  AIThreat_              2 columns
  AIComplex_             5 columns
  'Exact (ordinal/numeric)' 5 columns  ['Age', 'EdLevel', 'Employment', 'YearsCode', 'LearnCodeAI_flag']

Train set (5,800 rows):
  Intenders (1): 1,437   24.8%
  Rejectors (0): 4,363   75.2%

Test set (1,451 rows):
  Intenders (1):   360   24.8%
  Rejectors (0): 1,091   75.2%

Stratification confirmed — 25/75 ratio preserved in both splits.


---
## Section 10 — Save Outputs and Write Feature List

Four CSV files and one text file are written to `data/processed/`:

| File | Contents | Used by |
|---|---|---|
| `X_train.csv` | Training features (80%) | `03_modeling.ipynb` |
| `X_test.csv` | Test features (20%) | `03_modeling.ipynb` |
| `y_train.csv` | Training labels | `03_modeling.ipynb` |
| `y_test.csv` | Test labels | `03_modeling.ipynb` |
| `feature_names.txt` | One column name per line | `03_modeling.ipynb` (authoritative list) |

`df_intenders_rejectors.csv` (the full encoded model dataset before splitting)
is also saved for reference — useful if the clustering notebook needs the
full intender subgroup without re-running preprocessing.

`feature_names.txt` is the single source of truth for the modeling notebook.
It reads this file rather than hard-coding column names, so any future
preprocessing change automatically flows through to the model.

In [10]:
# ── Save train / test splits ───────────────────────────────────────────────────
X_train.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_test.to_csv( PROCESSED_DIR / 'X_test.csv',  index=False)
y_train.to_csv(PROCESSED_DIR / 'y_train.csv', index=False)
y_test.to_csv( PROCESSED_DIR / 'y_test.csv',  index=False)

# ── Save full encoded model dataset (pre-split) ────────────────────────────────
df_model.to_csv(DF_MODEL_PATH, index=False)

# ── Write feature names — one per line ────────────────────────────────────────
with open(FEATURE_NAMES_PATH, 'w', encoding='utf-8') as f:
    for col in X_train.columns:
        f.write(col + '\n')

# ── Summary ────────────────────────────────────────────────────────────────────
n_features = len(X_train.columns)

print(f'Files saved to {PROCESSED_DIR.resolve()}:')
for fname in ['X_train.csv', 'X_test.csv', 'y_train.csv', 'y_test.csv',
              'df_intenders_rejectors.csv', 'feature_names.txt']:
    fpath = PROCESSED_DIR / fname
    print(f'  {fname:<35}  {fpath.stat().st_size / 1024:>7.1f} KB')

print(f'\nTotal features going into the model: {n_features}')
print()
print('Feature list (first 20 shown — see feature_names.txt for the full list):')
for col in list(X_train.columns)[:20]:
    print(f'  {col}')
if n_features > 20:
    print(f'  ... and {n_features - 20} more')

Files saved to C:\Debjyoti\01 Personal\06 Claude\Data Science Projects\developer-ai-adoption-gap\data\processed:
  X_train.csv                           1671.8 KB
  X_test.csv                             419.7 KB
  y_train.csv                             17.0 KB
  y_test.csv                               4.3 KB
  df_intenders_rejectors.csv           20622.9 KB
  feature_names.txt                        2.1 KB

Total features going into the model: 60

Feature list (first 20 shown — see feature_names.txt for the full list):
  DevType_Architect
  DevType_Backend
  DevType_DesktopEnterprise
  DevType_FullStack
  DevType_Other
  DevType_Student
  AIFrustration_AlmostRight
  AIFrustration_DebuggingSlower
  AIFrustration_DontUseRegularly
  AIFrustration_HardToUnderstand
  AIFrustration_LessConfident
  AIFrustration_Other
  Industry_Computer Systems Design and Services
  Industry_Energy
  Industry_Fintech
  Industry_Government
  Industry_Healthcare
  Industry_Higher Education
  Industry_Insura

---
## Preprocessing Complete — Checklist

**Per CLAUDE.md success criteria:**

- [x] Raw data never modified — all cleaning done in code, `data/raw/` untouched
- [x] Multi-select columns exploded correctly — `DevType` (6 kept) and `AIFrustration` (6 kept), 5% threshold applied, originals dropped
- [x] Ordinal encoding applied to `Age` (1–6) and `EdLevel` (1–7)
- [x] `Employment` collapsed to 3 categories before encoding (0/1/2)
- [x] `WorkExp` dropped — r = 0.844 with `YearsCode`, multicollinearity risk (DECISIONS.md D-13)
- [x] Missing values imputed — `YearsCode` median, `Age` and `EdLevel` mode
- [x] `LearnCodeAI` converted to binary flag (1 = actively learned AI tooling)
- [x] Nominal columns encoded with `get_dummies(drop_first=True)`: `Industry`, `OrgSize`, `ICorPM`, `RemoteWork`, `PurchaseInfluence`, `AISent`, `AIThreat`, `AIComplex`
- [x] Binary target column `intender_flag` created (1 = intender, 0 = rejector)
- [x] Stratified 80/20 split confirmed — 24.8% intenders in both train and test
- [x] `feature_names.txt` written to `data/processed/` — authoritative feature list for modeling notebook
- [x] `df_valid.csv` saved (full four-group dataset for EDA reference)
- [x] `df_intenders_rejectors.csv` saved (full encoded model dataset, pre-split)
- [x] `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv` saved

**Next step:** `03_modeling.ipynb` — Random Forest → Logistic Regression → CART